# 09 AdaBoost

AdaBoost 是 boosting 的经典算法。它会连续训练很多弱分类器，每一轮更关注上一轮分错的样本。


## 0. 学习目标和阅读地图

AdaBoost 的核心是“错题本”：上一轮分错的样本，下一轮权重变大。

你需要掌握：

1. 样本权重如何变化。
2. 弱分类器权重 `alpha` 为什么由错误率决定。
3. 为什么 AdaBoost 对异常点敏感。
4. boosting 和 bagging 的差异。


## 1. 数学逻辑

AdaBoost 维护样本权重 `D_i`。弱分类器错误率：

$$err = \sum_i D_i \cdot I(h(x_i) \ne y_i)$$

弱分类器权重：

$$\alpha = \frac{1}{2}\log\frac{1-err}{err}$$

分错的样本权重会变大，分对的会变小。最终预测是加权投票：

$$H(x)=\text{sign}\left(\sum_t \alpha_t h_t(x)\right)$$


## 1.1 推导拆开看：错误率和 alpha

如果某个弱分类器错误率低，它应该在最终投票中更有话语权：

$$\alpha = \frac{1}{2}\log\frac{1-err}{err}$$

当 `err=0.5`，`alpha=0`，说明它和随机猜差不多；当 `err` 很小，`alpha` 很大。

样本权重更新：

$$D_i \leftarrow D_i\exp(-\alpha y_i h(x_i))$$

如果分类正确，`y_i h(x_i)=1`，权重变小；如果分类错误，`y_i h(x_i)=-1`，权重变大。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager


def setup_chinese_font():
    candidates = [
        'PingFang SC',
        'Heiti SC',
        'Songti SC',
        'Arial Unicode MS',
        'Noto Sans CJK SC',
        'Noto Sans SC',
        'SimHei',
        'Microsoft YaHei',
        'WenQuanYi Micro Hei',
    ]
    available = {font.name for font in font_manager.fontManager.ttflist}
    for font in candidates:
        if font in available:
            existing = [name for name in plt.rcParams['font.sans-serif'] if name != font]
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [font] + existing
            break
    else:
        print('Warning: no Chinese font found. Install Noto Sans CJK SC or SimHei if Chinese text is missing in plots.')
    plt.rcParams['axes.unicode_minus'] = False


setup_chinese_font()

from sklearn.datasets import make_classification
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y01 = make_classification(n_samples=250, n_features=2, n_redundant=0, n_informative=2,
                             n_clusters_per_class=1, class_sep=0.9, random_state=42)
y = np.where(y01 == 1, 1, -1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


## 1.2 AdaBoost 的训练动态

AdaBoost 不是一次性训练很多树，而是一轮接一轮训练。每一轮都依赖上一轮留下的样本权重。

这和随机森林完全不同：随机森林里的树可以并行训练，AdaBoost 的树天然是串行的。


In [ ]:
# 从零实现：用一维阈值 stump 做弱分类器

def fit_stump(X, y, sample_weight):
    best = {'err': float('inf')}
    for feature in range(X.shape[1]):
        for threshold in np.unique(X[:, feature]):
            for polarity in [1, -1]:
                pred = np.ones(len(y))
                pred[polarity * X[:, feature] < polarity * threshold] = -1
                err = np.sum(sample_weight[pred != y])
                if err < best['err']:
                    best = {'feature': feature, 'threshold': threshold, 'polarity': polarity, 'err': err, 'pred': pred}
    return best

sample_weight = np.ones(len(X_train)) / len(X_train)
learners = []
for t in range(12):
    stump = fit_stump(X_train, y_train, sample_weight)
    err = np.clip(stump['err'], 1e-12, 1 - 1e-12)
    alpha = 0.5 * np.log((1 - err) / err)
    sample_weight *= np.exp(-alpha * y_train * stump['pred'])
    sample_weight /= sample_weight.sum()
    learners.append((stump, alpha))
    print(f'round {t+1:2d} | err={err:.3f} | alpha={alpha:.3f}')


## 1.3 从零实现代码怎么读

从零实现使用非常弱的一维阈值分类器 stump：

1. `fit_stump` 找当前权重下错误率最低的一刀。
2. 根据错误率计算 `alpha`。
3. 分错样本权重变大。
4. 下一轮 stump 会更关注这些难样本。

打印的 `err` 和 `alpha` 可以帮助你观察每轮弱学习器的质量。


In [ ]:
base = DecisionTreeClassifier(max_depth=1, random_state=42)
try:
    model = AdaBoostClassifier(estimator=base, n_estimators=60, learning_rate=0.8, random_state=42)
except TypeError:
    model = AdaBoostClassifier(base_estimator=base, n_estimators=60, learning_rate=0.8, random_state=42)

model.fit(X_train, y_train)
pred = model.predict(X_test)
print('AdaBoost accuracy:', round(accuracy_score(y_test, pred), 3))

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 180),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=24)
plt.title('AdaBoost 决策边界')
plt.show()


In [ ]:
# 诊断：观察 boosting 轮数增加时的测试准确率
staged_acc = list(model.staged_score(X_test, y_test))
plt.plot(range(1, len(staged_acc) + 1), staged_acc)
plt.title('AdaBoost 轮数与测试准确率')
plt.xlabel('boosting round')
plt.ylabel('accuracy')
plt.show()
print('最佳测试准确率:', round(max(staged_acc), 3), '出现在第', int(np.argmax(staged_acc) + 1), '轮')


## 2.1 如何诊断 AdaBoost

如果测试准确率先升后降，说明后期可能开始追逐噪声或异常点。

AdaBoost 对错误标签尤其敏感，因为错误标签会一直被模型视为“难样本”，权重越来越高。


## 2. 常见误区

- AdaBoost 对异常值和错误标签比较敏感，因为它会越来越关注难分样本。
- 弱学习器通常要弱，常见选择是深度为 1 的决策树 stump。
- `learning_rate` 和 `n_estimators` 需要一起调。

## 3. 小实验

- 增加 `n_estimators`，观察训练是否过拟合。
- 调低 `learning_rate`，通常需要更多轮。
- 把弱学习器深度从 1 改到 2，观察边界变化。


## 5. 复习清单

- AdaBoost 是串行 boosting，不是并行 bagging。
- 每轮更关注上一轮分错的样本。
- 弱学习器错误率越低，最终投票权重越大。
- 对异常值和错误标签要小心。
